In [0]:
from pyspark.sql import functions as F

events = spark.table("urban_mobility.silver.trip_events_clean")
zones = spark.table("urban_mobility.silver.zones")

windowed = (
    events
    .withColumn(
        "window", F.window("event_timestamp", "5 minutes")
    )
    .groupBy("window", "pickup_zone_id")
    .agg(
        F.sum(F.when(F.col("event_type") == "TRIP_REQUESTED", 1).otherwise(0)).alias("trip_requests"),
        F.sum(F.when(F.col("event_type") == "TRIP_COMPLETED", 1).otherwise(0)).alias("completed_trips"),
        F.sum(F.when(F.col("event_type") == "TRIP_CANCELLED", 1).otherwise(0)).alias("cancelled_trips"),
        F.countDistinct(F.when(F.col("event_type") == "DRIVER_ASSIGNED", F.col("driver_id"))).alias("active_drivers"),
    )
    .withColumn("window_start", F.col("window.start"))
    .withColumn("window_end", F.col("window.end"))
    .drop("window")
    .withColumn(
        "demand_supply_ratio",
        F.when(F.col("active_drivers") > 0, F.round(F.col("trip_requests") / F.col("active_drivers"), 2))
         .otherwise(F.lit(None))
    )
    .withColumn(
        "demand_level",
        F.when(F.col("demand_supply_ratio").isNull(), "LOW")
         .when(F.col("demand_supply_ratio") < 1.0, "LOW")
         .when(F.col("demand_supply_ratio") < 2.0, "NORMAL")
         .when(F.col("demand_supply_ratio") < 4.0, "HIGH")
         .otherwise("CRITICAL")
    )
)

zone_demand = (
    windowed
    .join(
        zones.select(
            F.col("zone_id").alias("pickup_zone_id"),
            F.col("zone_name"),
            F.col("borough")
        ),
        "pickup_zone_id", "left"
    )
    .select(
        "window_start", "window_end",
        F.col("pickup_zone_id").alias("zone_id"),
        "zone_name", "borough",
        "trip_requests", "completed_trips", "cancelled_trips",
        "active_drivers", "demand_supply_ratio", "demand_level"
    )
)

zone_demand.write.mode("overwrite").format("delta").saveAsTable("urban_mobility.gold.zone_demand_5min")

result = spark.table("urban_mobility.gold.zone_demand_5min")
print("gold.zone_demand_5min rows:", result.count())
result.groupBy("demand_level").count().show()